# Práctica 3: Clasificadores Bayesianos
## Parte I — Clasificación de imágenes por color

Este notebook sigue el orden del PDF: **2.1 Extracción de características** y **2.2 Entrenamiento e implementación del clasificador QDA**.

Clases: plátanos, huevos, chiles y fondo.

In [ ]:
# 0. LIBRERÍAS
import os, re, glob, subprocess
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

print("Librerías cargadas correctamente.")

In [ ]:
# 0. SUBIR Y DESCOMPRIMIR EJERCICIO 1.RAR
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "unrar"], check=True)

uploaded = files.upload()
rar_files = [n for n in uploaded if n.lower().endswith(".rar")]
if not rar_files:
    raise FileNotFoundError("Sube Ejercicio 1.rar")

rar_path = rar_files[0]
data_root = Path("/content/ejercicio1")
data_root.mkdir(parents=True, exist_ok=True)
subprocess.run(["unrar", "x", "-o+", rar_path, str(data_root)], check=True)

all_files = sorted(p for p in data_root.rglob("*") if p.is_file())
print("Archivos encontrados:")
for p in all_files:
    print(p.relative_to(data_root))

## 2.1. Extracción de características
### a) Extracción de las máscaras de segmentación

Se segmentan plátanos, chiles y huevos. El fondo corresponde al complemento de las demás máscaras.

In [ ]:
# LOCALIZAR IMÁGENES
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
image_files = sorted(
    p for p in data_root.rglob("*")
    if p.is_file() and p.suffix.lower() in EXTS
)

print(f"Imágenes encontradas: {len(image_files)}")
for i, p in enumerate(image_files):
    print(f"{i:02d}: {p.relative_to(data_root)}")

In [ ]:
# MOSTRAR IMÁGENES
n_show = min(len(image_files), 12)
if n_show == 0:
    raise FileNotFoundError("No se encontraron imágenes.")

fig, axes = plt.subplots(
    int(np.ceil(n_show/3)), 3,
    figsize=(15, 4*int(np.ceil(n_show/3)))
)
axes = np.array(axes).reshape(-1)

for ax, path in zip(axes, image_files[:n_show]):
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(path.name)
    ax.axis("off")

for ax in axes[n_show:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# PARÁMETROS DE SEGMENTACIÓN HSV

# Chile: VERDE
GREEN_LOWER = np.array([35, 60, 30])
GREEN_UPPER = np.array([95, 255, 255])

# Plátano: AMARILLO
YELLOW_LOWER = np.array([18, 60, 60])
YELLOW_UPPER = np.array([40, 255, 255])

# Huevo: BLANCO
WHITE_LOWER = np.array([0, 0, 130])
WHITE_UPPER = np.array([179, 95, 255])


def segment_image(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # Chile = verde
    chile = cv2.inRange(
        hsv,
        GREEN_LOWER,
        GREEN_UPPER
    )

    # Plátano = amarillo
    platano = cv2.inRange(
        hsv,
        YELLOW_LOWER,
        YELLOW_UPPER
    )

    # Huevo = blanco
    huevo = cv2.inRange(
        hsv,
        WHITE_LOWER,
        WHITE_UPPER
    )

    # Evitar solapamiento entre clases
    platano = cv2.bitwise_and(
        platano,
        cv2.bitwise_not(chile)
    )

    huevo = cv2.bitwise_and(
        huevo,
        cv2.bitwise_not(chile | platano)
    )

    # Eliminar ruido
    kernel = np.ones((3, 3), np.uint8)

    chile = cv2.morphologyEx(
        chile,
        cv2.MORPH_OPEN,
        kernel
    )

    chile = cv2.morphologyEx(
        chile,
        cv2.MORPH_CLOSE,
        kernel
    )

    platano = cv2.morphologyEx(
        platano,
        cv2.MORPH_OPEN,
        kernel
    )

    platano = cv2.morphologyEx(
        platano,
        cv2.MORPH_CLOSE,
        kernel
    )

    huevo = cv2.morphologyEx(
        huevo,
        cv2.MORPH_OPEN,
        kernel
    )

    huevo = cv2.morphologyEx(
        huevo,
        cv2.MORPH_CLOSE,
        kernel
    )

    # Fondo = complemento de los objetos
    objetos = (
        (chile > 0) |
        (platano > 0) |
        (huevo > 0)
    )

    fondo = (~objetos).astype(np.uint8) * 255

    return {
        "plátano": platano,
        "chile": chile,
        "huevo": huevo,
        "fondo": fondo
    }

In [ ]:
# 2.1.b — MOSTRAR MÁSCARAS POR CLASE
def mostrar_mascaras(path):
    img_bgr = cv2.imread(str(path))
    masks = segment_image(img_bgr)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 5, figsize=(20,4))
    axes[0].imshow(img_rgb)
    axes[0].set_title("Imagen original")
    axes[0].axis("off")

    for ax, (clase, mask) in zip(axes[1:], masks.items()):
        ax.imshow(mask, cmap="gray")
        ax.set_title(clase)
        ax.axis("off")

    plt.suptitle(path.name)
    plt.tight_layout()
    plt.show()

for path in image_files:
    mostrar_mascaras(path)

## 2.1.c — Construcción del DataFrame

El DataFrame tendrá las cuatro columnas solicitadas: Canal 1 (rojo), Canal 2 (verde), Canal 3 (azul) y Clase. Cada píxel perteneciente a un objeto es un ejemplo.

In [ ]:
# 2.1.c — DATAFRAME RGB + CLASE
FEATURES = ["Canal 1 (rojo)", "Canal 2 (verde)", "Canal 3 (azul)"]
rows = []

# Solo las imágenes de entrenamiento (las de prueba no participan en el ajuste)
train_files = [p for p in image_files if "entrenamiento" in p.name.lower()]
print("Imágenes de entrenamiento:", [p.name for p in train_files])

# Recorrer las imágenes de entrenamiento
for path in train_files:
    img_bgr = cv2.imread(str(path))

    if img_bgr is None:
        continue

    # Convertir de BGR a RGB
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Obtener las máscaras de cada clase
    masks = segment_image(img_bgr)

    # Extraer los píxeles pertenecientes a cada clase
    for clase, mask in masks.items():
        ys, xs = np.where(mask > 0)
        pixels = img_rgb[ys, xs]

        for pixel in pixels:
            rows.append({
                FEATURES[0]: int(pixel[0]),
                FEATURES[1]: int(pixel[1]),
                FEATURES[2]: int(pixel[2]),
                "Clase": clase
            })

# Crear el DataFrame con todos los píxeles
df = pd.DataFrame(rows)

print("Dimensiones:", df.shape)

print("\nEjemplos de datos por clase:")
display(df.groupby("Clase", group_keys=False).head(3))

print("\nCantidad de píxeles por clase:")
display(df["Clase"].value_counts())

In [ ]:
plt.figure(figsize=(8,4))
df["Clase"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Clase")
plt.ylabel("Número de píxeles")
plt.title("Distribución de ejemplos por clase")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2.2. Entrenamiento e implementación del clasificador QDA
### a) Vector de medias, matriz de covarianza y probabilidad a priori

In [ ]:
# 2.2.a — ESTADÍSTICAS POR CLASE
estadisticas = {}

for clase in sorted(df["Clase"].unique()):
    Xc = df.loc[df["Clase"] == clase, FEATURES].to_numpy(dtype=float)
    media = np.mean(Xc, axis=0)
    cov = np.cov(Xc, rowvar=False)
    prior = len(Xc) / len(df)

    estadisticas[clase] = {
        "media": media,
        "covarianza": cov,
        "prior": prior
    }

    print("="*70)
    print("CLASE:", clase)
    print("Vector de medias:")
    print(media)
    print("\nMatriz de covarianza:")
    print(cov)
    print("\nProbabilidad a priori:", prior)

### b) Implementación de QDA

La función recibe una matriz de ejemplos RGB y retorna la clase estimada para cada píxel.

In [ ]:
# 2.2.b — QDA DESDE CERO
def qda_predict(X, estadisticas, regularizacion=1e-6):
    X = np.asarray(X, dtype=float)
    log_scores = []

    for clase, info in estadisticas.items():
        mu = info["media"]
        cov = info["covarianza"].copy()
        prior = info["prior"]

        cov_reg = cov + regularizacion*np.eye(cov.shape[0])
        sign, logdet = np.linalg.slogdet(cov_reg)

        if sign <= 0:
            raise np.linalg.LinAlgError(
                f"Covarianza no definida positiva para {clase}"
            )

        inv_cov = np.linalg.inv(cov_reg)
        diff = X - mu

        term = np.einsum("ij,jk,ik->i", diff, inv_cov, diff)
        score = -0.5*term - 0.5*logdet + np.log(prior)
        log_scores.append(score)

    log_scores = np.vstack(log_scores).T
    clases = list(estadisticas.keys())
    indices = np.argmax(log_scores, axis=1)

    return np.array([clases[i] for i in indices])

print(qda_predict(df[FEATURES].head(10).to_numpy(), estadisticas))

In [ ]:
# DETECTAR IMÁGENES DE PRUEBA
test_files = [
    p for p in image_files
    if re.search(r"(test|prueba|testing)", str(p).lower())
]

if test_files:
    print("Detectadas automáticamente:")
    for p in test_files:
        print("-", p.relative_to(data_root))
else:
    print("No se detectaron imágenes por nombre.")
    print("Índices disponibles:")
    for i, p in enumerate(image_files):
        print(i, p.relative_to(data_root))

    # Si hace falta, coloca aquí los índices de las 3 imágenes de prueba.
    TEST_INDICES = []
    test_files = [image_files[i] for i in TEST_INDICES]

print("Cantidad seleccionada:", len(test_files))

### c) Clasificación de las tres imágenes de prueba

In [ ]:
# 2.2.c — CLASIFICACIÓN QDA
CLASS_TO_ID = {
    "fondo": 0,
    "plátano": 1,
    "chile": 2,
    "huevo": 3
}

def clasificar_imagen_qda(path):
    img_bgr = cv2.imread(str(path))
    if img_bgr is None:
        raise ValueError(f"No se pudo leer {path}")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w, _ = img_rgb.shape

    X = img_rgb.reshape(-1,3).astype(float)
    pred = qda_predict(X, estadisticas)
    pred_ids = np.array([CLASS_TO_ID[p] for p in pred])

    return img_rgb, pred_ids.reshape(h,w)

resultados_prueba = {}

for path in test_files:
    original, pred = clasificar_imagen_qda(path)
    resultados_prueba[path.name] = (original, pred)

print("Clasificación terminada.")

In [ ]:
# VISUALIZAR RESULTADOS
from matplotlib.colors import ListedColormap

cmap = ListedColormap(["black", "yellow", "green", "white"])

for nombre, (original, pred) in resultados_prueba.items():
    fig, axes = plt.subplots(1,2,figsize=(12,5))

    axes[0].imshow(original)
    axes[0].set_title("Imagen de prueba")
    axes[0].axis("off")

    axes[1].imshow(pred, cmap=cmap, vmin=0, vmax=3)
    axes[1].set_title("Clasificación QDA")
    axes[1].axis("off")

    plt.suptitle(nombre)
    plt.tight_layout()
    plt.show()

## Análisis de resultados — Parte I

Después de ejecutar el notebook, analiza las máscaras y las tres imágenes clasificadas. Comenta qué clases se separan correctamente, dónde aparecen confusiones, cómo influye la iluminación y qué zonas presentan errores.

In [ ]:
print("PARTE I COMPLETADA\n")

print("2.1.a Máscaras: Las máscaras permiten separar plátanos, chiles, huevos y fondo. La segmentación funciona mejor en regiones con colores definidos.\n")

print("2.1.b Visualización de máscaras: Las clases se distinguen adecuadamente. Los principales errores aparecen en bordes, sombras y reflejos.\n")

print("2.1.c DataFrame RGB + clase: El DataFrame relaciona los valores RGB de cada píxel con su clase y sirve como datos de entrenamiento.\n")

print("2.2.a Medias, covarianzas y priors: Cada clase presenta una distribución diferente de valores RGB. El fondo tiene el mayor prior debido a que ocupa más píxeles.\n")

print("2.2.b QDA: QDA utiliza las medias, covarianzas y priors para calcular la probabilidad de cada clase y clasificar los píxeles.\n")

print("2.2.c Clasificación de las imágenes de prueba: Los plátanos, huevos y chiles se clasifican generalmente de forma correcta. Las confusiones aparecen en bordes, sombras, reflejos y zonas con diferente iluminación.Los objetos que no pertenecen a las clases entrenadas pueden ser asignados a la clase de color más similar.")